# Figure 1 — Benchmark design and data landscape

What goes *into* the benchmark: the three data regimes, the datasets, where they sit on the confounding axis, the simulation grid, and the method inventory. No results.

Each panel has its own cell and renders on its own, sized to suit it alone. The last cell assembles all of them at the submission size (180 x 230 mm) and writes the PDF.

The panel functions live in `figures/fig1.R` and are **sourced, not copied** — edit them there and re-run the cell, so the notebook and `snakemake` produce the same figure.

## Setup

In [ ]:
# Panels come from the script, so the notebook and the submitted figure cannot drift apart.
# main() is guarded, so sourcing runs nothing.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))
source("figures/fig1.R")

results_dir <- "results"

## Data

In [ ]:
datasets <- read.delim("data/DATASETS.tsv")
datasets <- datasets[datasets$track %in% c("real", "mock"), ]
scenarios <- read.delim("config/scenarios.tsv")
methods <- method_table()
methods <- methods[methods$role != "sensitivity", ]
confounding <- dataset_confounding_table(datasets, results_dir)

cat(nrow(datasets), "datasets,", nrow(scenarios), "scenarios,", nrow(methods), "methods\n")

## a. Design schematic

Three data regimes feed one wrapper interface; four metric families read its output. Drawn with elbow connectors so no two arrows cross.

In [ ]:
show_panel(panel_design(), 180, 55)

## b. Dataset landscape

Size against number of batches. Point area is the number of taxa; colour separates the sets that are scored from the three kept for calibration.

In [ ]:
show_panel(panel_landscape(datasets), 95, 72)

## c. Confounding axis

Where each real dataset falls on Cramer's V, against the five levels the simulation sweeps. mbqc at 0 is the fully crossed anchor.

In [ ]:
show_panel(panel_confounding(datasets, confounding, c(0, 0.3, 0.6, 0.9, 1)), 95, 72)

## d. Simulation parameter space

One strip per factor, one point per scenario: what the 213-scenario grid actually covers.

In [ ]:
show_panel(panel_simulation_space(scenarios), 180, 68)

## e. Method inventory

What each method needs and what it returns.

In [ ]:
show_panel(panel_inventory(methods), 170, 92)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 230 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
figure <- (panel_design() /
           (panel_landscape(datasets) | panel_confounding(datasets, confounding,
                                                          c(0, 0.3, 0.6, 0.9, 1))) /
           panel_simulation_space(scenarios) /
           panel_inventory(methods)) +
  plot_layout(heights = c(1, 1.1, 1.1, 1.3)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 9, face = "bold"))

save_figure(figure, "results/figures/fig1.pdf", 180, 230)
show_panel(figure, 180, 230, scale = 1.5)